# Raw window-neighbourhood source files

Load Karol's original MEROPS dictionaries/matrices and preview every original PICS and qPISA workbook. Each worksheet preview shows its first three source rows with every column visible. Use the repository `.venv` kernel. Downloads and rebuilding the existing MEROPS CSV are optional.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
from zipfile import ZipFile

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "assets/tools/proteolysis_data_manager").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from the pep-compass repository or set ROOT to its path.")

DATA_ROOT = ROOT / "data/proteolysis/window_neighbourhood_datasets"
RAW = DATA_ROOT / "raw"
MEROPS_ROOT = ROOT / "data/proteolysis/window_neighbourhood_datasets/raw/merops"
MEROPS_INDEX_PATH = MEROPS_ROOT / "index.json"
SCRIPT_PYTHON = ROOT / ".venv/bin/python"
if not SCRIPT_PYTHON.is_file():
    SCRIPT_PYTHON = Path(sys.executable)
RUN_DOWNLOAD = False
RUN_MERGE = False
PREVIEW_ROWS = 10
XLSX_PREVIEW_ROWS = 3
MEROPS_DATASET_INDEX = 35
MEROPS_ROW = 0
MMP_PICS_FILE = "mmc5.xlsx"
MMP_PICS_SHEET = "MMP1_T"
DPP4_QPISA_FILE = "44320_2024_71_MOESM2_ESM.xlsx"
DPP4_QPISA_SHEET = "Peptide_Levels_dpf-3_dpp4"

pd.set_option("display.max_rows", PREVIEW_ROWS)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 60)

## Acquire original files (optional)

Set `RUN_DOWNLOAD = True` to copy Karol's original MEROPS tree and download publication artifacts. Leave it `False` to read the existing files.

In [2]:
DOWNLOAD_SCRIPT = ROOT / "assets/scripts/downloads/proteolysis_data_manager/window_neighbourhood_data/download_all.py"
if RUN_DOWNLOAD:
    subprocess.run([str(SCRIPT_PYTHON), str(DOWNLOAD_SCRIPT), "--data-root", str(DATA_ROOT)], cwd=ROOT, check=True)
if not (RAW / "manifest.json").is_file():
    raise FileNotFoundError(f"Raw files are missing from {RAW}. Set RUN_DOWNLOAD = True and run this cell again.")
print("Raw files:", RAW)

Raw files: /home/max/repositories/pep-compass/data/proteolysis/window_neighbourhood_datasets/raw


## Load the original MEROPS dictionaries and matrices

Read `raw/merops/index.json`, then use its dataset folder and `codes.json` row index to load the corresponding original NPY and CSV files.

In [3]:
merops_source = MEROPS_ROOT
merops_index = json.loads(MEROPS_INDEX_PATH.read_text())
print("Loaded MEROPS index:", MEROPS_INDEX_PATH)
display(pd.DataFrame(merops_index["by_index"].values()).head(PREVIEW_ROWS))

merops_entry = merops_index["by_index"][str(MEROPS_DATASET_INDEX)]
merops_directory = MEROPS_ROOT / merops_entry["folder"]
merops_codes_path = merops_directory / "codes.json"
merops_codes = json.loads(merops_codes_path.read_text())
display(pd.DataFrame(merops_codes["peptidases"]).head(PREVIEW_ROWS))

merops_counts = np.load(merops_directory / "matrices_counts.npy", allow_pickle=False)  # (N, P, A)
merops_logprob = np.load(merops_directory / "matrices_logprob.npy", allow_pickle=False)  # (N, P, A)
merops_peptidase = next(item for item in merops_codes["peptidases"] if item["row"] == MEROPS_ROW)
print("Original matrix:", merops_entry["folder"], merops_peptidase)
merops_count_matrix = merops_counts[MEROPS_ROW]  # (P, A)
merops_logprob_matrix = merops_logprob[MEROPS_ROW]  # (P, A)
display(pd.DataFrame(merops_count_matrix, index=merops_codes["positions"], columns=merops_codes["amino_acids"]))
display(pd.DataFrame(merops_logprob_matrix, index=merops_codes["positions"], columns=merops_codes["amino_acids"]))
merops_original_csv = pd.read_csv(merops_directory / f'{merops_peptidase["code"]}.csv', dtype=object, keep_default_na=False)
display(merops_original_csv)

Loaded MEROPS index: /home/max/repositories/pep-compass/data/proteolysis/window_neighbourhood_datasets/raw/merops/index.json
Original matrix: 35_human_serum {'row': 0, 'code': 'C111.001', 'name': 'coagulation factor XIIIa', 'n_cleavages': 1, 'serum_category': 'coagulation'}


,dataset_index,folder,scientific,role,apex_index,label,taxids,n_peptidases
0,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0.0,A. baumannii ATCC 19606,[sp002236],35
1,1,01_e_coli_atcc_11775,Escherichia coli,pathogen,1.0,E. coli ATCC 11775,"[sp000403, sp024561, sp026615, sp027897, sp030023, sp030...",91
2,2,02_e_coli_aig221,Escherichia coli,pathogen,2.0,E. coli AIG221,"[sp000403, sp024561, sp026615, sp027897, sp030023, sp030...",91
3,3,03_e_coli_aig222,Escherichia coli,pathogen,3.0,E. coli AIG222,"[sp000403, sp024561, sp026615, sp027897, sp030023, sp030...",91
4,4,04_k_pneumoniae_atcc_13883,Klebsiella pneumoniae,pathogen,4.0,K. pneumoniae ATCC 13883,"[sp000567, sp029925]",69
5,5,05_p_aeruginosa_pa01,Pseudomonas aeruginosa,pathogen,5.0,P. aeruginosa PA01,"[sp000835, sp003781, sp018769, sp030450]",50
6,6,06_p_aeruginosa_pa14,Pseudomonas aeruginosa,pathogen,6.0,P. aeruginosa PA14,"[sp000835, sp003781, sp018769, sp030450]",50
7,7,07_s_aureus_atcc_12600,Staphylococcus aureus,pathogen,7.0,S. aureus ATCC 12600,"[sp000130, sp000970, sp002830, sp011283, sp011285, sp018...",55
8,8,08_s_aureus_atcc_baa_1556_mrsa,Staphylococcus aureus,pathogen,8.0,S. aureus (ATCC BAA-1556) - MRSA,"[sp000130, sp000970, sp002830, sp011283, sp011285, sp018...",55
9,9,09_vancomycin_resistant_e_faecalis_atcc_700802,Enterococcus faecalis,pathogen,9.0,vancomycin-resistant E. faecalis ATCC 700802,"[sp000382, sp000383, sp019459, sp032648]",48


,row,code,name,n_cleavages,serum_category
0,0,C111.001,coagulation factor XIIIa,1,coagulation
1,1,M01.001,aminopeptidase N,42,metalloproteases_plasma
2,2,M02.001,angiotensin-converting enzyme peptidase unit 1,7,metalloproteases_plasma
3,3,M02.004,angiotensin-converting enzyme peptidase unit 2,13,metalloproteases_plasma
4,4,M02.006,angiotensin-converting enzyme-2,14,metalloproteases_plasma
5,5,M10.001,matrix metallopeptidase-1,83,metalloproteases_plasma
6,6,M10.002,matrix metallopeptidase-8,114,metalloproteases_plasma
7,7,M10.003,matrix metallopeptidase-2,3417,metalloproteases_plasma
8,8,M10.004,matrix metallopeptidase-9,370,metalloproteases_plasma
9,9,M10.005,matrix metallopeptidase-3,2468,metalloproteases_plasma


,A,C,D,E,F,G,...,R,S,T,V,W,Y
P4,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P3,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P2,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P1,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P1p,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P2p,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P3p,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0
P4p,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0


,A,C,D,E,F,G,...,R,S,T,V,W,Y
P4,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,...,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732
P3,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,...,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732
P2,-3.091043,-3.091043,-3.091043,-3.091043,-3.091043,-3.091043,...,-3.091043,-3.091043,-3.091043,-3.091043,-3.091043,-3.091043
P1,-3.091043,-3.091043,-3.091043,-1.992430,-3.091043,-3.091043,...,-3.091043,-3.091043,-3.091043,-3.091043,-3.091043,-3.091043
P1p,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,...,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732
P2p,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,...,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732
P3p,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,...,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732
P4p,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,...,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732,-2.995732


,position,A,C,D,E,F,...,R,S,T,V,W,Y
0,P4,0,0,0,0,0,...,0,0,0,0,0,0
1,P3,0,0,0,0,0,...,0,0,0,0,0,0
2,P2,0,0,0,0,0,...,0,0,0,0,0,0
3,P1,0,0,0,1,0,...,0,0,0,0,0,0
4,P1p,0,0,0,0,0,...,0,0,0,0,0,0
5,P2p,0,0,0,0,0,...,0,0,0,0,0,0
6,P3p,0,0,0,0,0,...,0,0,0,0,0,0
7,P4p,0,0,0,0,0,...,0,0,0,0,0,0


## Load the existing MEROPS CSV

The CSV joins Karol's dictionaries and matrix cells. Set `RUN_MERGE = True` only to regenerate it with the existing script; source files remain unchanged.

In [4]:
MERGE_SCRIPT = ROOT / "assets/scripts/integration/proteolysis_data_manager/window_neighbourhood_data/merge_merops.py"
if RUN_MERGE:
    subprocess.run([str(SCRIPT_PYTHON), str(MERGE_SCRIPT), "--data-root", str(DATA_ROOT)], cwd=ROOT, check=True)

merops_merged_csv = DATA_ROOT / "processed/merops_matrices.csv"
if merops_merged_csv.is_file():
    merops_merged = pd.read_csv(merops_merged_csv, dtype={"apex_index": "Int64"}, float_precision="round_trip", low_memory=False)
    print("Merged matrix cells:", len(merops_merged))
    display(merops_merged.head(PREVIEW_ROWS))
else:
    print("CSV is not present. Set RUN_MERGE = True to generate it:", merops_merged_csv)

Merged matrix cells: 284000


,dataset_index,folder,scientific,role,apex_index,label,...,position_index,position,amino_acid_index,amino_acid,count,logprob
0,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,0,A,4.0,-1.754019
1,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,1,C,0.0,-3.951244
2,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,2,D,0.0,-3.951244
3,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,3,E,0.0,-3.951244
4,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,4,F,0.0,-3.951244
5,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,5,G,1.0,-2.852631
6,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,6,H,1.0,-2.852631
7,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,7,I,0.0,-3.951244
8,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,8,K,1.0,-2.852631
9,0,00_a_baumannii_atcc_19606,Acinetobacter baumannii,pathogen,0,A. baumannii ATCC 19606,...,0,P4,9,L,3.0,-2.005333


## Preview every MMP/PICS workbook

Every original `.xlsx` file is opened, including `mmc4.xlsx` for MMP/PICS. For each sheet, display the first three physical rows (`header=None`) with all columns and complete cell text. Descriptive rows and original column headers remain visible. These are raw worksheet previews, without choosing a data header or filtering rows.

In [5]:
pics_workbook_previews = {}
for workbook_path in sorted((RAW / "mmp_pics").glob("*.xlsx")):
    with pd.ExcelFile(workbook_path) as workbook:
        print("File:", workbook_path.name, "| Sheets:", len(workbook.sheet_names))
        for sheet_name in workbook.sheet_names:
            preview = pd.read_excel(
                workbook, sheet_name=sheet_name, header=None,
                nrows=XLSX_PREVIEW_ROWS, dtype=object, keep_default_na=False,
            )
            pics_workbook_previews[(workbook_path.name, sheet_name)] = preview
            print("Sheet:", sheet_name)
            with pd.option_context(
                "display.max_rows", None, "display.max_columns", None,
                "display.max_colwidth", None,
            ):
                display(preview.head(XLSX_PREVIEW_ROWS))

File: mmc4.xlsx | Sheets: 3
Sheet: Index
Sheet: Supplementary Table 1
Sheet: Supplementary Table 2
File: mmc5.xlsx | Sheets: 10
Sheet: Supplementary Table 3
Sheet: MMP1_T
Sheet: MMP2_T
Sheet: MMP3_T
Sheet: MMP7_T
Sheet: MMP8_T
Sheet: MMP9_T
Sheet: MMP12_T
Sheet: MMP13_T
Sheet: MMP14_T
File: mmc6.xlsx | Sheets: 10
Sheet: Supplementary Table 4
Sheet: MMP1_G
Sheet: MMP2_G
Sheet: MMP3_G
Sheet: MMP7_G
Sheet: MMP8_G
Sheet: MMP9_G
Sheet: MMP12_G
Sheet: MMP13_G
Sheet: MMP14_G
File: mmc7.xlsx | Sheets: 11
Sheet: Supplementary Table 5
Sheet: Overview
Sheet: MMP1
Sheet: MMP2
Sheet: MMP3
Sheet: MMP7
Sheet: MMP8
Sheet: MMP9
Sheet: MMP12
Sheet: MMP13
Sheet: MMP14
File: mmc8.xlsx | Sheets: 10
Sheet: Supplementary Table 6
Sheet: MMP1
Sheet: MMP2
Sheet: MMP3
Sheet: MMP7
Sheet: MMP8
Sheet: MMP9
Sheet: MMP12
Sheet: MMP13
Sheet: MMP14
File: mmc9.xlsx | Sheets: 3
Sheet: Supplementary Table 7
Sheet: X! Tandem
Sheet: Mascot


,0,1,2,3
0,,,,
1,,,,"Ulrich Eckharda,‡, Pitter F. Huesgena,1‡, Oliver Schillinga,2‡, Caroline L. Bellaca,3#, Georgina S. Butlera#,"
2,,,,"Jennifer H. Coxa,4#, Antoine Dufoura#, Verena Goebelera#, Reinhild Kappelhoffa#, Ulrich auf dem Kellera,5#, Theo Kleina#, Philipp F. Langea#,"


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,ST-1,Active site specificity profiling datasets of human Matrix Metalloproteinases (MMPs):,,,,,,,,,,,,,,"Domain organization of human MMPs and respective UniProt-, Pfam-, Merops-, and TopFIND weblinks .",,,,,Index
1,,,,,,,,,,,,,,,,,,,,,
2,MMP,Other name,#aa,Signal Pep,,,Pro Pep,,,Cat-Dom,,,Hpx,,,Note,UniProt,Pfam,Merops,TopFIND,


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51
0,ST-2,Active site specificity profiling datasets of human Matrix Metalloproteinases (MMPs):,,,,,,,,,,,,,,,,,,,,,,,,,,Key residues and secondary structure elements of human MMPs analyzed.,,,,,,,,,,,,,,,,,,,,,,,,Index
1,,,(A) Active Site Residues,,,,,,,,,(B) Secondary Structure Elements of MMP catalytic domains.,,,,,,,,,,,,,,,,,,,,,(C) Length of Secondary Structure Elements (#) and distance to previous element (Δ),,,,,,,,,,,,,,,,,,,
2,MMP,,H,E,..,H,..,H,..,M,,Nter,,beta-1,,alpha-1,,beta-2,,beta-3,,beta-4,,beta-5,,alpha-2,,alpha-3,,Cter,,,Nter,,Δ,#,Δ,#,Δ,#,Δ,#,Δ,#,Δ,#,Δ,#,Δ,#,Cter,


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,,,,,,,,,,,,,,,,
1,,,,,,,,,,,,,,"Ulrich Eckharda,‡, Pitter F. Huesgena,1‡, Oliver Schillinga,2‡, Caroline L. Bellaca,3#, Georgina S. Butlera#,",,Proteomic Identiﬁcation of protease Cleavage Sites (PICS) - Experimental Workflow
2,,,,,,,,,,,,,,"Jennifer H. Coxa,4#, Antoine Dufoura#, Verena Goebelera#, Reinhild Kappelhoffa#, Ulrich auf dem Kellera,5#, Theo Kleina#, Philipp F. Langea#,",,


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
0,MMP13,Active site specificity profiling datasets of human Matrix Metalloproteinases (MMPs):,,,,,,,,,,,,MMP activity profiling using quenched fluorescence (QF) peptides.,,,,,,,,,Index
1,,,,,,,,,,,,,,,,,,,,,,,
2,,,,,,,,,,,,,,,,Analysis Round 1,,Graph Template: fluorescence (a.u.) vs. time (s),,,,,


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
0,MMP14,Active site specificity profiling datasets of human Matrix Metalloproteinases (MMPs):,,,,,,,,,,,,MMP activity profiling using quenched fluorescence (QF) peptides.,,,,,,,,,Index
1,,,,,,,,,,,,,,,,,,,,,,,
2,,,,,,,,,,,,,,,,Analysis Round 1,,Graph Template: fluorescence (a.u.) vs. time (s),,,,,


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,,,,,,,,,,,,,,
1,,,,,,,,,,,,,,"Ulrich Eckharda,‡, Pitter F. Huesgena,1‡, Oliver Schillinga,2‡, Caroline L. Bellaca,3#, Georgina S. Butlera#,"
2,,,,,,,,,,,,,,"Jennifer H. Coxa,4#, Antoine Dufoura#, Verena Goebelera#, Reinhild Kappelhoffa#, Ulrich auf dem Kellera,5#, Theo Kleina#, Philipp F. Langea#,"


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
0,Active site specificity profiling datasets of human Matrix Metalloproteinases (MMPs):,,,,,,,,,,,,Database search settings for successful PICS data analysis - X!Tandem,,,,,,,,,,Index
1,,,,,,,,,,,,,,,,,,,,,,,
2,"<note type=""input"" label=""spectrum, parent monoisotopic mass error plus"">200</note>",,,,,,,,,,,,,,,,,"For more details, please refer to:",,,,,http://www.thegpm.org/TANDEM/api/index.html


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
0,Active site specificity profiling datasets of human Matrix Metalloproteinases (MMPs):,,,,,,,,,,,,Database search settings for successful PICS data analysis - Mascot,,,,,,,,,,Index
1,,,,,,,,,,,,,,,,,,,,,,,
2,,,,,,,,,,,,,,,,,"For more details, please refer to:",,,,,,http://www.matrixscience.com/help/search_field_help.html


## Load one selected MMP/PICS sheet

After reviewing the workbook previews above, choose `MMP_PICS_FILE` and `MMP_PICS_SHEET` in the configuration cell to load that complete sheet.

In [6]:
mmp_pics_files = sorted(path.name for path in (RAW / "mmp_pics").glob("*.xlsx"))
print("Available files:", mmp_pics_files)
with pd.ExcelFile(RAW / "mmp_pics" / MMP_PICS_FILE) as workbook:
    print("Available sheets:", workbook.sheet_names)
    mmp_pics_raw = pd.read_excel(workbook, sheet_name=MMP_PICS_SHEET, header=None, dtype=object, keep_default_na=False)
print("Loaded worksheet shape:", mmp_pics_raw.shape)
display(mmp_pics_raw.head(PREVIEW_ROWS))

mmp_pics_article_xml = (RAW / "mmp_pics/PMC4777984.1.xml").read_text()
with ZipFile(RAW / "mmp_pics/mmc10.zip") as archive:
    mmp_pics_webpics_files = archive.namelist()
print("WebPICS archive members:", len(mmp_pics_webpics_files))
display(mmp_pics_webpics_files[:PREVIEW_ROWS])

Available files: ['mmc4.xlsx', 'mmc5.xlsx', 'mmc6.xlsx', 'mmc7.xlsx', 'mmc8.xlsx', 'mmc9.xlsx']
Available sheets: ['Supplementary Table 3', 'MMP1_T', 'MMP2_T', 'MMP3_T', 'MMP7_T', 'MMP8_T', 'MMP9_T', 'MMP12_T', 'MMP13_T', 'MMP14_T']
Loaded worksheet shape: (1000, 27)
WebPICS archive members: 1073


,0,1,2,3,4,5,...,21,22,23,24,25,26
0,hMMP1-T_1%,Active site specificity profiling datasets of human Matr...,,,,,...,,,,,,Index
1,,,,,,,...,,,,,,
2,n[89.01],,PICS Output,,PICS Output - unique,,...,,,,UniProt_Human_1310,,
3,A.n[89.01]HNVAAQGK[156.13].Y,,SSVSMIASRKPT,,SSVSMIASRKPT,,...,P5',P6',,,counts,%
4,P.n[89.01]VNSHPQR.S,,DMIVC-AYAHEL,,DMIVC-AYAHEL,,...,22,21,,G,1497164,6.511197
5,D.n[89.01]C[160.03]HTAHIAC[160.03]K[156.13].F,,YQEPIPCEQLVT,,YQEPIPCEQLVT,,...,23,29,,A,1594886,6.936192
6,V.n[89.01]INGGSHAGNK[156.13].L,,Y-PV-DCHTAHI,,Y-PV-DCHTAHI,,...,25,25,,V,1383482,6.016792
7,M.n[89.01]VTPGHAC[160.03]TQK[156.13].F,,QPTPDSCIISMV,,QPTPDSCIISMV,,...,12,8,,I,1004174,4.367173
8,S.n[89.01]HK[156.13]DDSPDLPK[156.13].L,,I-SN--CTTNCL,,I-SN--CTTNCL,,...,39,28,,L,2257963,9.819927
9,F.n[89.01]VM[147.04]GVNHEK[156.13].Y,,LGTTYSCVGVFQ,,LGTTYSCVGVFQ,,...,9,5,,M,490375,2.132651


['Info_WebPICS-Results-of-MMP-Specificity-Profiling.txt',
 'WebPICS_hMMP12_G_1%/',
 'WebPICS_hMMP12_G_1%/._DrawParameters.txt',
 'WebPICS_hMMP12_G_1%/cleavage-sites.txt',
 'WebPICS_hMMP12_G_1%/DrawParameters.txt',
 'WebPICS_hMMP12_G_1%/EnzymeParameters.txt',
 'WebPICS_hMMP12_G_1%/global_heatmap_occurence.png',
 'WebPICS_hMMP12_G_1%/global_heatmap_ratio.png',
 'WebPICS_hMMP12_G_1%/global_percent.csv',
 'WebPICS_hMMP12_G_1%/global_total.csv']

## Preview every DPP4/qPISA workbook

Every original `.xlsx` file is opened, including `mmc4.xlsx` for MMP/PICS. For each sheet, display the first three physical rows (`header=None`) with all columns and complete cell text. Descriptive rows and original column headers remain visible. These are raw worksheet previews, without choosing a data header or filtering rows.

In [7]:
qpisa_workbook_previews = {}
for workbook_path in sorted((RAW / "dpp4_qpisa").glob("*.xlsx")):
    with pd.ExcelFile(workbook_path) as workbook:
        print("File:", workbook_path.name, "| Sheets:", len(workbook.sheet_names))
        for sheet_name in workbook.sheet_names:
            preview = pd.read_excel(
                workbook, sheet_name=sheet_name, header=None,
                nrows=XLSX_PREVIEW_ROWS, dtype=object, keep_default_na=False,
            )
            qpisa_workbook_previews[(workbook_path.name, sheet_name)] = preview
            print("Sheet:", sheet_name)
            with pd.option_context(
                "display.max_rows", None, "display.max_columns", None,
                "display.max_colwidth", None,
            ):
                display(preview.head(XLSX_PREVIEW_ROWS))

File: 44320_2024_71_MOESM2_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV1
Sheet: Peptide_Levels_dpf-3_dpp4
File: 44320_2024_71_MOESM3_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV2
Sheet: dpp4_modelParams
File: 44320_2024_71_MOESM4_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV3
Sheet: dpf-3_modelParams
File: 44320_2024_71_MOESM5_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV4
Sheet: Peptide_Levels_dpf-3
File: 44320_2024_71_MOESM6_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV5
Sheet: dpf-3_modelParams
File: 44320_2024_71_MOESM7_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV6
Sheet: Product peptide intensities
File: 44320_2024_71_MOESM8_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV7
Sheet: Cryo-EM parameters
File: 44320_2024_71_MOESM9_ESM.xlsx | Sheets: 2
Sheet: Legend Data set EV8
Sheet: primers


,0
0,Data set EV1 - List of peptides detected and quantified in input as well as in DPF-3 and DPP4 digested condition. Levels are log2-transformed and mean centered.


,0,1,2,3,4,5
0,Peptide squence,Status,DPF-3,DPP4,No enzyme control (for comparing with DPF-3 digested samples),No enzyme control (for comparing with DPP4 digested samples)
1,AAAAAAAAAAAAAAAGAGAGAK,Tryptic+,-2.546241,-3.15339,-2.524359,-2.496487
2,AAAAAAAAAAGAAGGR,Non-Tryptic+,-1.997493,-2.042909,-1.930733,-1.947317


,0
0,Dataset EV2 - Model parameters used for predicting the turnover of peptides upon digestion by DPP4


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41
0,Amino acid (single letter code),P1,P2:A,P2:C,P2:D,P2:E,P2:F,P2:G,P2:H,P2:I,P2:K,P2:L,P2:M,P2:N,P2:P,P2:Q,P2:R,P2:S,P2:T,P2:V,P2:W,P2:Y,P1':A,P1':C,P1':D,P1':E,P1':F,P1':G,P1':H,P1':I,P1':K,P1':L,P1':M,P1':N,P1':P,P1':Q,P1':R,P1':S,P1':T,P1':V,P1':W,P1':Y
1,A,2.6329,0.1324,0.3758,-1.4089,-0.1953,0.4243,-0.728,-0.2231,0.3711,0.7305,0.2823,0.0416,-0.47,-0.0495,0.4582,0.5742,-0.5258,-0.5078,0.3471,-0.2534,0.6244,0.3553,0.5459,-0.1238,-0.2404,0.1991,-0.2935,0.2808,-0.3478,0.6608,0.3076,0.572,0.4108,-2.6116,0.3447,0.3828,0.3412,-0.8287,-0.5813,0.0608,0.5651
2,C,0.0314,0.0027,0.0466,-0.0163,0.052,-0.02,-0.0061,-0.0283,-0.0048,NA,0.0194,-0.0135,-0.0122,NA,0.0299,NA,-0.0079,-0.0198,-0.0073,NA,-0.0144,0.0213,-0.0208,-0.0291,-0.0111,-0.005,0,0.0685,0.0715,NA,-0.0177,-0.0138,0.0385,-0.0204,-0.012,NA,-0.0125,-0.0152,-0.0034,NA,-0.039


...

,0,1,2,3,4,5,6,7,8
0,,Peptide name: HAE,,Peptide name: DAE,,Peptide name: HAP,,Peptide name: DAP,
1,condition,HAEGTFTSDVSR (substrate),EGTFTSDVSR (product),DAEGTFTSDVSR (substrate),EGTFTSDVSR (product),HAPGTFTSDVSR (substrate),PGTFTSDVSR (product),DAPGTFTSDVSR (substrate),PGTFTSDVSR (product)
2,Buffer only. Replicate A. 0 min,865850000,3400800,102290000,1353400,2255000000,150790,3938200000,47049000


,0
0,"Dataset EV7 - Collection, refinement and processing of DPF-3 cryo-EM data."


,0,1
0,,
1,,Apo C. elegans DPF-3
2,Data collection and processing,


,0
0,Dataset EV8 - list of oligonucleotides used in this study


,0,1,2
0,Number,Sequence,comment
1,DPF-3 Fr_pAC8_N,CTCTACTTCCAGGGAGGCGGCCGCttcaatttttatcagtttct,Oligo couple (DPF-3 Fr_pAC8_N & DPF-3 Re_pAC8_C) to amplify the dpf-3 cDNA and clone it into the pAC8 vector by gibson cloning method.
2,DPF-3 Re_pAC8_C,TTAATTAAGCGGCCGTCCGGGGGTACCTTAtcaaagcgggccttgtctt,


## Load one selected DPP4/qPISA sheet

After reviewing the workbook previews above, choose `DPP4_QPISA_FILE` and `DPP4_QPISA_SHEET` in the configuration cell to load that complete sheet.

In [8]:
dpp4_qpisa_files = sorted(path.name for path in (RAW / "dpp4_qpisa").glob("*.xlsx"))
print("Available files:", dpp4_qpisa_files)
with pd.ExcelFile(RAW / "dpp4_qpisa" / DPP4_QPISA_FILE) as workbook:
    print("Available sheets:", workbook.sheet_names)
    dpp4_qpisa_raw = pd.read_excel(workbook, sheet_name=DPP4_QPISA_SHEET, header=None, dtype=object, keep_default_na=False)
print("Loaded worksheet shape:", dpp4_qpisa_raw.shape)
display(dpp4_qpisa_raw.head(PREVIEW_ROWS))

dpp4_qpisa_article_xml = (RAW / "dpp4_qpisa/PMC11612144.1.xml").read_text()

Available files: ['44320_2024_71_MOESM2_ESM.xlsx', '44320_2024_71_MOESM3_ESM.xlsx', '44320_2024_71_MOESM4_ESM.xlsx', '44320_2024_71_MOESM5_ESM.xlsx', '44320_2024_71_MOESM6_ESM.xlsx', '44320_2024_71_MOESM7_ESM.xlsx', '44320_2024_71_MOESM8_ESM.xlsx', '44320_2024_71_MOESM9_ESM.xlsx']
Available sheets: ['Legend Data set EV1', 'Peptide_Levels_dpf-3_dpp4']
Loaded worksheet shape: (53500, 6)


,0,1,2,3,4,5
0,Peptide squence,Status,DPF-3,DPP4,No enzyme control (for comparing with DPF-3 digested sam...,No enzyme control (for comparing with DPP4 digested samp...
1,AAAAAAAAAAAAAAAGAGAGAK,Tryptic+,-2.546241,-3.15339,-2.524359,-2.496487
2,AAAAAAAAAAGAAGGR,Non-Tryptic+,-1.997493,-2.042909,-1.930733,-1.947317
3,AAAAAAAAAPAAAATAPTTAATTAATAAQ,Tryptic+,-1.25117,-1.456395,-1.112593,-1.336112
4,AAAAAAAGDSDSWDADAFSVEDPVRK,Non-Tryptic+,-1.873291,-1.645871,-1.738154,-1.732066
5,AAAAAADPNAAWAAYYSH,Non-Tryptic+,-0.853053,-3.870517,0.832043,0.773883
6,AAAAAAGAASGLPGPVAQGLK,Non-Tryptic+,-0.901355,-0.733458,-0.914158,-0.917373
7,AAAAAWEEPSSGNGTAR,Tryptic+,-2.547247,-4.589404,-1.825033,-2.016079
8,AAAAELSLLEK,Non-Tryptic+,0.933135,1.022274,0.965691,0.944782
9,AAAALINFTEDCPK,Non-Tryptic+,-1.05754,-3.527752,1.385485,1.401045
